In [0]:
sources = "licenses"
control_timestamp = "updated_at"
min_control_timestamp = '2026-04-01T14:11:27.060Z'

In [0]:
%run "../_micellanous/utils/init_environment_base"

In [0]:
%run "../_micellanous/utils/init_bronze_environment"

In [0]:
if spark.catalog.tableExists(bronze_full_table_name):
    max_ts = spark.sql(f"""
                    SELECT MAX(updated_at) as max_ts
                    FROM {bronze_full_table_name}
                    """).first()["max_ts"]    
    print(f"[Incremental mode] last updated_at found {max_ts}")
else:
    max_ts = min_control_timestamp
    print(f"[Full-historical mode] last updated_at set to {max_ts}")

In [0]:
import requests  
import json

url = "https://data.cityofnewyork.us/resource/ptev-4hud.json"

limit = 50000
offset = 0

all_data = []

while True:

    params = {
        "$select": "*, :id, :created_at, :updated_at",
        "$where": f":updated_at > '{max_ts}'",
        "$order": ":updated_at ASC, :id ASC",
        "$limit": limit,
        "$offset": offset
    }

    response = requests.get(url, params=params)

    print(f"Url final: {response.url}")
    print(f"Status: {response.status_code}")

    if response.status_code != 200:
        raise Exception(f"Error when connecting with the API: {response.status_code} - {response.text}")

    data = response.json()

    if len(data) == 0:
        print("No more data")
        break

    all_data.extend(data)

    print(f"Offset {offset} - registros recogidos: {len(data)}")

    if len(data) < limit:
        print("All data for this call was collected")
        print(f"Total licenses collected: {len(all_data)}")
        break

    offset += limit

In [0]:
rows = []

for license in data:
    socrata_id = license.get(":id")
    application_id = license.get("application_id")
    license_number = license.get("license_number")
    json_raw = json.dumps(license, ensure_ascii=False)
    created_at = license.get(":created_at")
    updated_at = license.get(":updated_at")

    rows.append({"socrata_id" : socrata_id,
                "application_id" : application_id,
                "license_number" : license_number,
                "json_raw": json_raw,
                "created_at" : created_at,
                "updated_at" : updated_at})

In [0]:
df_base = spark.createDataFrame(rows)
df_base.createOrReplaceTempView("vw_base")

In [0]:
schema_ddl = spark.sql("""
                       SELECT  schema_of_json_agg(json_raw) AS schema
                       FROM vw_base
                       """).first()["schema"]

In [0]:
import json
from pyspark.sql import functions as sf 
from pyspark.sql.functions import current_date, current_timestamp

df_bronze = df_base\
            .withColumn("json_struct", sf.from_json(df_base["json_raw"], schema_ddl))\
            .withColumn("ts_ingestion", current_timestamp())\
            .withColumn("dt_ingestion", current_date())\
            .select("socrata_id", "application_id", "license_number", "json_raw", "json_struct", "created_at", "updated_at")

df_bronze.createOrReplaceTempView("vw_bronze")            

In [0]:
summary = {
    "inserted" : 0,
    "updated" : 0,
    "deleted" : 0
}

loaded_count = df_bronze.count()

if not spark.catalog.tableExists(bronze_full_table_name):
    spark.sql(f"""
              CREATE TABLE {bronze_full_table_name}
              USING DELTA
              TBLPROPERTIES (
                  delta.enableChangeDataFeed= true,
                  delta.enableDeletionVectors= true
              )
              AS
              SELECT * FROM vw_bronze
              """)
    print("Table newly created")
    print(f"Data appended to {bronze_full_table_name}")
    summary["inserted"] = loaded_count
else:
    df_bronze.write\
    .mode("append")\
    .option("mergeSchema", "true")\
    .format("delta")\
    .saveAsTable(bronze_full_table_name)
    print(f"Data appended to {bronze_full_table_name}")
    summary["inserted"] = loaded_count
    print("RESUMEN BRONZE")
    print(f"Inserts : {summary['inserted']}")
    print(f"Updates : {summary['updated']}")
    print(f"Deleted : {summary['deleted']}")
    print("=" * 50)